# Week 20, Day 2: Your own modules

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w020d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
print(__name__)

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('units.py', 'w', encoding='utf-8') as f:
    f.write('"""Converting between units of measurement.\n\n    >>> import units\n    >>> units.c_to_f(100)\n    212.0\n"""\n\nKM_PER_MILE = 1.609344\n\ndef c_to_f(celsius):\n    """Celsius to Fahrenheit."""\n    return celsius * 9 / 5 + 32\n\ndef f_to_c(fahrenheit):\n    """Fahrenheit to Celsius."""\n    return (fahrenheit - 32) * 5 / 9\n\ndef miles_to_km(miles):\n    return _round(miles * KM_PER_MILE)\n\ndef km_to_miles(km):\n    return _round(km / KM_PER_MILE)\n\ndef _round(x):\n    # one place to change how results are rounded\n    return round(x, 3)\n')

In [ ]:
import units

print(units.c_to_f(37), units.f_to_c(212))
print(units.miles_to_km(26.2), units.km_to_miles(5))
print(units.__doc__.splitlines()[0])

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shopping.py', 'w', encoding='utf-8') as f:
    f.write('"""A shopping basket."""\n\ndef total(prices):\n    return round(sum(prices), 2)\n\nprint("shopping.py: __name__ is", repr(__name__))\n\nif __name__ == "__main__":\n    print("Demo: total of 1.5 and 2.25 is", total([1.5, 2.25]))\n')

In [ ]:
import shopping

print("main program: __name__ is", repr(__name__))
print(shopping.total([3.99, 1.01]))

> **Key idea.** A module should define things, not do things, when imported. Put demo code, tests and the program's start inside `if __name__ == "__main__":`, which is only true when the file is run directly.

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('shop/__init__.py', 'w', encoding='utf-8') as f:
    f.write('"""A small shop: prices, baskets and receipts."""\nfrom .basket import Basket_total\n')
with open('shop/prices.py', 'w', encoding='utf-8') as f:
    f.write('PRICES = {"tea": 2.50, "jam": 3.10, "bread": 1.20}\nVAT_RATE = 0.2\n\ndef price_of(item):\n    return PRICES[item]\n')
with open('shop/basket.py', 'w', encoding='utf-8') as f:
    f.write('from .prices import price_of, VAT_RATE\n\ndef Basket_total(items):\n    """Total for a list of item names, with VAT."""\n    net = sum(price_of(i) for i in items)\n    return round(net * (1 + VAT_RATE), 2)\n')
with open('shop/receipt.py', 'w', encoding='utf-8') as f:
    f.write('from . import basket\n\ndef receipt(items):\n    lines = [f"{i:<8}" for i in items]\n    lines.append(f"TOTAL   {basket.Basket_total(items):.2f}")\n    return "\\n".join(lines)\n')

In [ ]:
import shop.prices
from shop.receipt import receipt
from shop import Basket_total

print(shop.prices.price_of("jam"))
print(receipt(["tea", "bread", "bread"]))
print(Basket_total(["jam"]))

### Exercise: Write a module

Write the code for a module `temperature.py` with: a docstring; `c_to_f(c)` and `f_to_c(f)` (rounded to 1 decimal place); a private helper `_check(t)` that raises `ValueError` if a Celsius temperature is below absolute zero (-273.15), called by `c_to_f`; and a demo inside `if __name__ == "__main__":` that prints `c_to_f(100)`. The checks save your code as `temperature.py` and import it.

In [ ]:
"""___"""

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_module():\n    \"works as a module\"\n    write_file(\"temperature.py\", source)\n    import importlib, sys\n    sys.modules.pop(\"temperature\", None)\n    t = importlib.import_module(\"temperature\")\n    assert t.c_to_f(100) == 212.0 and t.f_to_c(98.6) == 37.0 and t.c_to_f(-40) == -40.0\n    try:\n        t.c_to_f(-300)\n        assert False, \"below absolute zero should raise ValueError\"\n    except ValueError:\n        pass\n    assert t.__doc__ and t.__doc__.strip() and t.__doc__.strip() != \"___\"\n    assert hasattr(t, \"_check\")\ndef test_main():\n    \"the demo only runs as a program\"\n    expect_output(\"212.0\")\n    assert '__name__ == \"__main__\"' in source or \"__name__ == '__main__'\" in source")

### Exercise: Guard the demo

`stats_tools.py` prints a demo every time it's imported, which clutters every program that uses it. Rewrite the code below so the demo only runs when the file is run directly. The program must still print the two demo lines when run.

In [ ]:
def mean(xs):
    return sum(xs) / len(xs)

def spread(xs):
    return max(xs) - min(xs)

print("demo mean:", mean([2, 4, 9]))
print("demo spread:", spread([2, 4, 9]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"guarded\"\n    expect_output(\"demo mean: 5.0\\ndemo spread: 7\")\n    write_file(\"stats_tools.py\", source)\n    import importlib, sys, io, contextlib\n    sys.modules.pop(\"stats_tools\", None)\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        m = importlib.import_module(\"stats_tools\")\n    assert buf.getvalue() == \"\", \"importing it still prints\"\n    assert m.mean([1, 3]) == 2")

### Exercise: Use a package

Using the `shop` package on this page, set `a` to the price of bread (from `shop.prices`), `b` to the receipt text for `["jam", "tea"]` (from `shop.receipt`), and `c` to the VAT rate. Use at least one `from ... import ...` line and at least one plain `import` line.

In [ ]:
a = ___
b = ___
c = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 1.2 and b == \"jam     \\ntea     \\nTOTAL   6.72\" and c == 0.2\n    assert \"from shop\" in source and any(l.strip().startswith(\"import shop\") for l in source.splitlines())")

### Exercise: Build a package

Make a package `geometry` with two modules, by writing their code into files with the starter's `write_file` helper: `geometry/__init__.py` (which makes `area` available as `from geometry import area`), `geometry/shapes.py` with `circle(r)` and `rectangle(w, h)` returning areas (circle to 2 decimal places, using `math.pi`), and `geometry/calc.py` with `area(kind, *sizes)` that calls the right shape function (`"circle"` or `"rectangle"`), importing `shapes` with a **relative** import. Do all of it in a function `build()` that writes the three files.

In [ ]:
import os

def write_file(name, text):
    """Write text to a file, making its folder first if needed."""
    os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
    with open(name, "w") as f:
        f.write(text)

def build():
    write_file("geometry/__init__.py", """___""")
    write_file("geometry/shapes.py", """___""")
    write_file("geometry/calc.py", """___""")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"the package works\"\n    import sys, importlib\n    for name in list(sys.modules):\n        if name == \"geometry\" or name.startswith(\"geometry.\"):\n            del sys.modules[name]\n    build()\n    importlib.invalidate_caches()\n    from geometry import area\n    from geometry.shapes import circle\n    assert area(\"circle\", 1) == 3.14 and area(\"rectangle\", 3, 4) == 12 and circle(2) == 12.57\n    calc = read_file(\"geometry/calc.py\")\n    assert \"from .\" in calc or \"from .shapes\" in calc")

### Exercise: The shape of a script

Rewrite this script into the standard shape: a function `read_scores(text)` that turns the text into a list of whole numbers, a function `summary(scores)` that returns the string shown, and a `main()` that does the printing, called from an `if __name__ == "__main__":` block. Nothing else at the top level except the `TEXT` constant and the functions.

In [ ]:
TEXT = "72 45 90 61"

scores = [int(t) for t in TEXT.split()]
print(f"{len(scores)} scores, best {max(scores)}, average {sum(scores) / len(scores):.1f}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"shape\"\n    expect_output(\"4 scores, best 90, average 67.0\")\n    assert read_scores(\"1 2 3\") == [1, 2, 3]\n    assert summary([10, 20]) == \"2 scores, best 20, average 15.0\"\n    import ast\n    top = [type(n).__name__ for n in ast.parse(source).body]\n    assert set(top) <= {\"Assign\", \"FunctionDef\", \"If\", \"Expr\", \"Import\", \"ImportFrom\"}\n    assert top.count(\"Expr\") <= 1")

### Bug hunt: It asks for input when imported

`ages.py` asks for a name the moment it's imported, so the main program waits for typing. Fix `ages.py` (shown in the editor) so it only asks when run directly. The program run by the checks imports it and should print `adult` without asking anything.

In [ ]:
def category(age):
    return "child" if age < 18 else "adult"

name = input("Name? ")
print(name, "is", category(int(input("Age? "))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_import():\n    \"no input when imported\"\n    write_file(\"ages.py\", source)\n    import sys, importlib, builtins\n    sys.modules.pop(\"ages\", None)\n    real = builtins.input\n    def nope(*a):\n        raise AssertionError(\"ages.py asked for input when imported\")\n    builtins.input = nope\n    try:\n        m = importlib.import_module(\"ages\")\n    finally:\n        builtins.input = real\n    assert m.category(30) == \"adult\"")

### Bug hunt: A package that can't find its own module

Inside the `money` package, `convert.py` does `import rates`, which fails: `rates` isn't on `sys.path`, it's inside the package. Fix the import in `convert.py` (shown in the starter, which writes it out) to a relative one so the program prints `12.5`.

In [ ]:
# Run this cell first: it creates the files used below.
with open('money/__init__.py', 'w', encoding='utf-8') as f:
    f.write('\n')
with open('money/rates.py', 'w', encoding='utf-8') as f:
    f.write('RATES = {"GBP": 1.0, "EUR": 1.25}\n')

In [ ]:
write_text = """import rates

def to(amount, currency):
    return amount * rates.RATES[currency]
"""
with open("money/convert.py", "w") as f:
    f.write(write_text)

from money.convert import to
print(to(10, "EUR"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"12.5\"\n    expect_output(\"12.5\")")

### Puzzle: The shape of a program

Arrange the lines into a well-shaped program. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # __name__ inside helpers.py when another program does `import helpers`
b = ___   # __name__ in a file you run directly
c = ___   # the file that makes a folder into a package (its name)
d = ___   # in a package, what `from .prices import vat` means: the prices module in the same ___ (one word)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"helpers\" and b == \"__main__\" and c == \"__init__.py\" and d == \"package\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: split a program into a package

Here's a working one-file program for a book club's reading log. Split it into a package called `club`, by writing these files with `write_file` from a function `split_up()`:

- `club/records.py`: `parse(text)` (turns the log text into a list of `(member, book, pages)` tuples).
- `club/stats.py`: `pages_by_member(records)` (a dictionary) and `top_reader(records)` (the member with the most pages, ties alphabetically). It must not print anything.
- `club/report.py`: `report(text)`, returning the report string, using the other two modules through **relative** imports.
- `club/__init__.py`: makes `report` importable as `from club import report`.

The original program, for reference:

```
LOG = """Ada,Kindred,264
Ben,Dune,412
Ada,Dawn,248
Cy,Emma,474
Ben,Emma,120"""

totals = {}
for line in LOG.splitlines():
    member, book, pages = line.split(",")
    totals[member] = totals.get(member, 0) + int(pages)
best = sorted(totals, key=lambda m: (-totals[m], m))[0]
print("\\n".join(f"{m}: {totals[m]}" for m in sorted(totals)))
print(f"Top reader: {best}")
```

In [ ]:
import os

def write_file(name, text):
    """Write text to a file, making its folder first if needed."""
    os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
    with open(name, "w") as f:
        f.write(text)

def split_up():
    write_file("club/__init__.py", """___""")
    write_file("club/records.py", """___""")
    write_file("club/stats.py", """___""")
    write_file("club/report.py", """___""")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LOG = \"Ada,Kindred,264\\nBen,Dune,412\\nAda,Dawn,248\\nCy,Emma,474\\nBen,Emma,120\"\ndef fresh():\n    import sys, importlib\n    for name in list(sys.modules):\n        if name == \"club\" or name.startswith(\"club.\"):\n            del sys.modules[name]\n    split_up()\n    importlib.invalidate_caches()\ndef test_parts():\n    \"modules\"\n    fresh()\n    from club.records import parse\n    from club.stats import pages_by_member, top_reader\n    recs = parse(LOG)\n    assert recs[0] == (\"Ada\", \"Kindred\", 264) and len(recs) == 5\n    assert pages_by_member(recs) == {\"Ada\": 512, \"Ben\": 532, \"Cy\": 474}\n    assert top_reader(recs) == \"Ben\"\ndef test_report():\n    \"report\"\n    fresh()\n    from club import report\n    assert report(LOG) == \"Ada: 512\\nBen: 532\\nCy: 474\\nTop reader: Ben\"\n    rep = read_file(\"club/report.py\")\n    assert \"from .\" in rep and \"print(\" not in read_file(\"club/stats.py\")")

## Warm-up

Every module has a `__name__`. When you run a file directly, as every program on this site is run, its `__name__` is `"__main__"`. When the same file is imported, `__name__` is the module's own name. That difference is today's most useful trick.

## Writing a module

There's nothing special about a module: write functions in a `.py` file, and another file in the same folder can import them. A good module has:

- a **docstring** at the top saying what it's for;
- **public** functions and constants that other code is meant to use;
- **private** helpers, named with a leading `_`, that are its own business;
- **no** code at the top level that does things (printing, asking for input, changing files) when it's imported.

The module is now reusable: any program that needs unit conversions imports it, and if a bug is found, it's fixed in one place for all of them.

## __name__ and __main__

When you write a module, you often want a little demo or some quick tests at the bottom, to run when you work on it. But when another program imports the module, that demo shouldn't run. The answer:

```
def c_to_f(celsius):
    return celsius * 9 / 5 + 32

if __name__ == "__main__":
    # only runs when this file is run directly, not when it's imported
    print("Testing c_to_f...")
    print(c_to_f(100))
```

- Run the file directly: `__name__` is `"__main__"`, so the demo runs.
- Import it: `__name__` is `"units"` (or whatever the module is called), so the demo is skipped.

Imported, `shopping.py` saw its `__name__` as `'shopping'`, so its demo didn't run (the unguarded `print` did, which is exactly why top-level actions in a module are a bad idea).

Many programs put their whole top level in a `main()` function and finish with:

```
def main():
    ...

if __name__ == "__main__":
    main()
```

That's the standard shape of a Python program you'll see everywhere, and you'll use it for every script from now on: it means every file can be imported (for testing or reuse) without side effects.

## Packages

A **package** is a folder of modules with a file called `__init__.py` in it. The folder name becomes the package name, and its modules are reached with dots:

Inside the package, modules import each other with **relative imports**: `from .prices import price_of` means "from the `prices` module in the same package as me", and `from . import basket` means "the `basket` module in my package". Relative imports keep working if the whole package is renamed or moved.

`__init__.py` runs when the package is first imported. It can be empty, or it can import the most useful names so users can write the shorter `from shop import Basket_total` instead of `from shop.basket import Basket_total`. (The capital B is a poor name: functions should be lower case, as you'll fix in today's practice.)

## Splitting a program

Good places to split a program:

- **Data and its rules** (what a record is, how to validate it) in one module.
- **Calculations** (pure functions: totals, statistics, searches) in another.
- **Input and output** (reading files, printing reports, asking questions) in another.
- **The program itself** (`main()`, which ties them together) in a small top file.

The test of a good split: each module could be described in one sentence, and changing how reports are printed shouldn't mean touching the calculations. Modules that **don't** print or ask for input are also much easier to test, which is what Week 24 is about.

## Module style

- File and module names are short, lower case, with underscores if needed: `units.py`, `bank_rules.py`.
- Imports go at the top, after the docstring, in three groups: standard library, third-party, your own.
- Don't name a module after a standard one (`random.py`, `json.py`): it hides the real one.
- Constants (`KM_PER_MILE`) in capitals, near the top.
- If you use `from module import *` anywhere (in an `__init__.py`, say), list the public names in `__all__ = ["c_to_f", "f_to_c"]` so only those are exported.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the longest program you've written in this course and sketch how you'd split it into modules: write down each module's name and its one-sentence job. Tomorrow: a tour of three standard modules you'll use constantly, `math`, `random` and `statistics`.